# mPES - Colab GPU Retraining (DQN / RDQN / Transformer)

Este notebook usa las copias de `h1/` y `utils/` almacenadas en Google Drive.
Sube ambas carpetas bajo `MyDrive/mPES/` y ejecuta las celdas en orden.

Usalo unicamente para `dqn`, `rdqn` o `tr`. Los ensembles no se entrenan:
se optimizan desde `launch.ipynb` usando los modelos guardados en `h1`.

In [ ]:
"""Configure GPU retraining from h1 and utils stored on Drive."""
import glob
import json
import os
import shutil
import subprocess
import time

DRIVE_ROOT  = '/content/drive/MyDrive/mPES'
DRIVE_H1    = os.path.join(DRIVE_ROOT, 'h1')
DRIVE_UTILS = os.path.join(DRIVE_ROOT, 'utils')
WORKSPACE   = '/content/mPES'
H_DIR       = os.path.join(WORKSPACE, 'h1')
PKG         = 'rdqn'  # dqn | rdqn | tr
NUM_EPISODES           = 0  # 0 = best_params.json; >0 overrides
BEST_PARAMS_DRIVE_DATE = ''  # YYYY-MM-DD, or '' for repo input

if PKG not in ('dqn', 'rdqn', 'tr'):
    raise ValueError(f'PKG must be dqn|rdqn|tr, got {PKG!r}')
if not os.path.isdir(DRIVE_H1):
    raise FileNotFoundError(f'Upload h1 to: {DRIVE_H1}')
if not os.path.isfile(os.path.join(DRIVE_UTILS, 'config', 'requirements.txt')):
    raise FileNotFoundError(f'Upload utils to: {DRIVE_UTILS}')
os.environ.update({
    'DRIVE_H1': DRIVE_H1,
    'DRIVE_UTILS': DRIVE_UTILS,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'REQ_FILE': os.path.join(WORKSPACE, 'utils', 'config', 'requirements.txt'),
    'WORKSPACE_DIR': WORKSPACE,
    'PKG': PKG,
    'NUM_EPISODES': str(NUM_EPISODES),
    'BEST_PARAMS_DRIVE_DATE': BEST_PARAMS_DRIVE_DATE,
    'MPES_USE_GPU': '1',
})
print(f'[CELL 1] PKG={PKG!r} NUM_EPISODES={NUM_EPISODES}')

In [ ]:
# ==========================================================================
# Cell 2 - MOUNT DRIVE + VERIFY GPU
# ==========================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

try:
    _proc = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False)
    _ok = _proc.returncode == 0 and 'GPU 0' in _proc.stdout
    _msg = _proc.stdout.strip() if _ok else (_proc.stdout + _proc.stderr)
except FileNotFoundError:
    _ok, _msg = False, 'nvidia-smi not installed (CPU runtime)'
if not _ok:
    raise RuntimeError(
        'No GPU detected (' + _msg.strip() + '). '
        'Switch the runtime: Runtime -> Change runtime type -> T4 GPU, '
        'then Runtime -> Run all again.'
    )
print('[OK]', _msg)

In [ ]:
# Copy h1 and utils to local Colab storage for faster file access.
local_h1 = os.path.join(WORKSPACE, 'h1')
local_utils = os.path.join(WORKSPACE, 'utils')
for local_path in (local_h1, local_utils):
    if os.path.isdir(local_path):
        shutil.rmtree(local_path)
os.makedirs(WORKSPACE, exist_ok=True)
shutil.copytree(DRIVE_H1, local_h1)
shutil.copytree(DRIVE_UTILS, local_utils)

subprocess.run(
    ['bash', os.path.join(local_h1, 'general', 'colab', 'setup_colab.sh')],
    check=True,
    env=os.environ.copy(),
)
print(f'[CELL 3] Local copies: {local_h1} and {local_utils}')

In [ ]:
PKG = os.environ['PKG']
PKG_DIR_NAME = {'dqn': 'pes_dqn', 'rdqn': 'pes_rdqn', 'tr': 'pes_trf'}[PKG]
REPO_INPUTS = os.path.join(H_DIR, 'ml', PKG_DIR_NAME, 'inputs')
REPO_BP = os.path.join(REPO_INPUTS, 'best_params.json')

drive_date = os.environ.get('BEST_PARAMS_DRIVE_DATE', '').strip()
if drive_date:
    drive_bp = os.path.join(
        DRIVE_ROOT, PKG_DIR_NAME,
        f'{drive_date}_BAYESIAN_OPT', f'best_params_{drive_date}.json',
    )
    if not os.path.isfile(drive_bp):
        raise FileNotFoundError(f'Drive best_params not found: {drive_bp}')
    os.makedirs(REPO_INPUTS, exist_ok=True)
    shutil.copy2(drive_bp, REPO_BP)
else:
    if not os.path.isfile(REPO_BP):
        raise FileNotFoundError(f'No best_params.json found at {REPO_BP}')

with open(REPO_BP, 'r', encoding='utf-8') as file:
    best_params = json.load(file)
print(f'[CELL 4] best_params={REPO_BP} trial={best_params.get("best_trial_number")}')

In [ ]:
PKG = os.environ['PKG']
PKG_DIR_NAME = {'dqn': 'pes_dqn', 'rdqn': 'pes_rdqn', 'tr': 'pes_trf'}[PKG]
REPO_BP = os.path.join(H_DIR, 'ml', PKG_DIR_NAME, 'inputs', 'best_params.json')
MODULE = {
    'dqn': 'ml.pes_dqn.ext.train_dqn',
    'rdqn': 'ml.pes_rdqn.ext.train_rdqn',
    'tr': 'ml.pes_trf.ext.train_transformer',
}[PKG]

num_override = int(os.environ.get('NUM_EPISODES', '0'))
if num_override > 0:
    num_episodes = num_override
else:
    with open(REPO_BP, 'r', encoding='utf-8') as file:
        best_params = json.load(file)
    num_episodes = int(best_params['hyperparameters']['num_episodes'])
print(f'[CELL 5] episodes={num_episodes}')

_script = f'''set -uo pipefail
cd "$H_DIR"
source /content/mpes_env.sh
python -u -m {MODULE} {num_episodes}
'''
env = os.environ.copy()
env.update({'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'})
start = time.time()
process = subprocess.Popen(
    _script, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
)
assert process.stdout is not None
for line in process.stdout:
    print(line.rstrip(), flush=True)
process.wait()
if process.returncode != 0:
    raise RuntimeError(f'training exited with code {process.returncode}')
print(f'[CELL 5] Done in {(time.time() - start) / 60:.1f} min')

In [ ]:
PKG = os.environ['PKG']
PKG_DIR_NAME = {'dqn': 'pes_dqn', 'rdqn': 'pes_rdqn', 'tr': 'pes_trf'}[PKG]
TRAIN_TAG = {'dqn': 'DQN_TRAIN', 'rdqn': 'RDQN_TRAIN', 'tr': 'TRF_TRAIN'}[PKG]
REPO_INPUTS = os.path.join(H_DIR, 'ml', PKG_DIR_NAME, 'inputs')
DRIVE_INPUTS = os.path.join(DRIVE_ROOT, PKG_DIR_NAME)
os.makedirs(DRIVE_INPUTS, exist_ok=True)

runs = sorted(glob.glob(os.path.join(REPO_INPUTS, f'*_{TRAIN_TAG}')))
if not runs:
    raise FileNotFoundError(f'No *_{TRAIN_TAG} directory found under {REPO_INPUTS}')
source_run = runs[-1]
destination = os.path.join(DRIVE_INPUTS, os.path.basename(source_run))
if os.path.exists(destination):
    shutil.rmtree(destination)
shutil.copytree(source_run, destination)

for name in os.listdir(REPO_INPUTS):
    source = os.path.join(REPO_INPUTS, name)
    if os.path.isfile(source) and name.endswith(('.keras', '.npy')):
        shutil.copy2(source, os.path.join(DRIVE_INPUTS, name))
print(f'[CELL 6] Copied {source_run} -> {destination}')